# Dataset balancer (reales + sintéticas)

- Cuenta **instancias** (anotaciones YOLO, una línea = un bbox) en el dataset real y en el sintético.
- Objetivo por clase: \(M = \max_c \text{instancias reales}(c)\).
- Se eligen imágenes sintéticas (sin repetir) hasta cubrir, en la medida de lo posible, el déficit \(M - \text{real}(c)\) por clase.
- Si `augmentation_ratio` \(> 0\) (p. ej. `0.2`), se añaden **`round(augmentation_ratio * M)`** instancias sintéticas **por clase**, de nuevo de forma equilibrada, con imágenes nuevas respecto a la fase anterior.

Requisitos: estructura YOLO en reales (`images/{split}`, `labels/{split}`), sintéticas con `images/`, `labels/` y `prompts.csv` (índice en la primera columna, como en `dataset_transfer`).

In [1]:
from __future__ import annotations

import os
import random
import shutil
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Set, Tuple

import pandas as pd
import yaml
from tqdm import tqdm

# --- Parámetros ---
ruta_reales = "./datasets/VOC"
ruta_sinteticas = "./datasets/sVOC_FLUXFP8"
dataset_final = "./datasets/voc_balanced"

real_split = "train"  # subcarpeta images/{split} y labels/{split}
augmentation_ratio: float = 0.0  # 0 desactiva; 0.2 -> +round(0.2 * M) inst. sint./clase
random_state = 42
use_symlinks = True  # False para copiar archivos
data_yaml_name = "data.yaml"
prompts_csv_name = "prompts.csv"

if not (0.0 <= augmentation_ratio <= 1.0):
    raise ValueError("augmentation_ratio debe estar en [0, 1]")

In [2]:
def load_id_to_name(yaml_path: Path) -> Dict[int, str]:
    with open(yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    names = cfg["names"]
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    if isinstance(names, list):
        return {i: str(n) for i, n in enumerate(names)}
    raise ValueError("data.yaml: 'names' debe ser dict o lista")


def count_yolo_label_file(label_path: Path) -> Dict[int, int]:
    counts: Dict[int, int] = defaultdict(int)
    if not label_path.is_file():
        return counts
    text = label_path.read_text(encoding="utf-8", errors="ignore").strip()
    if not text:
        return counts
    for line in text.splitlines():
        parts = line.strip().split()
        if not parts:
            continue
        cls = int(float(parts[0]))
        counts[cls] += 1
    return counts


def count_real_dataset(labels_dir: Path) -> Tuple[Dict[int, int], int]:
    per_class: Dict[int, int] = defaultdict(int)
    label_files = list(labels_dir.glob("*.txt"))
    for lp in label_files:
        c = count_yolo_label_file(lp)
        for k, v in c.items():
            per_class[k] += v
    return per_class, len(label_files)


@dataclass
class SynthImage:
    idx: int
    img_path: Path
    lbl_path: Path
    counts: Dict[int, int]


def find_image_for_id(images_dir: Path, idx: int) -> Optional[Path]:
    for ext in (".png", ".jpg", ".jpeg", ".webp"):
        p = images_dir / f"{idx}{ext}"
        if p.is_file():
            return p
    return None


def build_synth_pool(synth_root: Path, prompts_name: str) -> List[SynthImage]:
    csv_path = synth_root / prompts_name
    df = pd.read_csv(csv_path)
    if "Unnamed: 0" in df.columns:
        ids = df["Unnamed: 0"].astype(int).tolist()
    elif str(df.columns[0]).startswith("Unnamed"):
        ids = df.iloc[:, 0].astype(int).tolist()
    else:
        df = pd.read_csv(csv_path, index_col=0)
        ids = df.index.astype(int).tolist()

    images_dir = synth_root / "images"
    labels_dir = synth_root / "labels"
    pool: List[SynthImage] = []
    missing_img = 0
    missing_lbl = 0
    for idx in ids:
        img_path = find_image_for_id(images_dir, int(idx))
        lbl_path = labels_dir / f"{idx}.txt"
        if img_path is None:
            missing_img += 1
            continue
        if not lbl_path.is_file():
            missing_lbl += 1
            continue
        counts = dict(count_yolo_label_file(lbl_path))
        pool.append(SynthImage(int(idx), img_path, lbl_path, counts))
    if missing_img or missing_lbl:
        print(f"Aviso: sintéticas omitidas — sin imagen: {missing_img}, sin etiqueta: {missing_lbl}")
    return pool


def marginal_gain(counts: Dict[int, int], deficit: Dict[int, int], class_ids: Iterable[int]) -> int:
    g = 0
    for cid in class_ids:
        need = deficit.get(cid, 0)
        if need <= 0:
            continue
        g += min(counts.get(cid, 0), need)
    return g


def apply_selection(
    pool: List[SynthImage],
    class_ids: List[int],
    deficit: Dict[int, int],
    forbidden: Set[int],
    rng: random.Random,
) -> List[int]:
    """Greedy: elige índices de SynthImage.idx hasta que deficit[c] <= 0 para todo c o no hay ganancia."""
    deficit = {c: int(deficit[c]) for c in class_ids}
    chosen: List[int] = []
    pool_work = [s for s in pool if s.idx not in forbidden]
    rng.shuffle(pool_work)

    while any(deficit[c] > 0 for c in class_ids):
        best: Optional[SynthImage] = None
        best_gain = 0
        best_tie: List[SynthImage] = []
        for s in pool_work:
            if s.idx in forbidden or s.idx in chosen:
                continue
            g = marginal_gain(s.counts, deficit, class_ids)
            if g > best_gain:
                best_gain = g
                best_tie = [s]
            elif g == best_gain and g > 0:
                best_tie.append(s)
        if best_gain <= 0 or not best_tie:
            break
        pick = rng.choice(best_tie)
        chosen.append(pick.idx)
        forbidden.add(pick.idx)
        for cid, v in pick.counts.items():
            if cid in deficit:
                deficit[cid] = max(0, deficit[cid] - v)
    return chosen


def summarize_from_images(
    pool_by_idx: Dict[int, SynthImage], chosen_ids: List[int], class_ids: List[int]
) -> Dict[int, int]:
    total: Dict[int, int] = {c: 0 for c in class_ids}
    for i in chosen_ids:
        s = pool_by_idx[i]
        for c in class_ids:
            total[c] += s.counts.get(c, 0)
    return total


def link_pair(
    src_img: Path,
    src_lbl: Path,
    dst_img_dir: Path,
    dst_lbl_dir: Path,
    use_symlinks: bool,
) -> None:
    dst_img = dst_img_dir / src_img.name
    dst_lbl = dst_lbl_dir / f"{src_img.stem}.txt"
    if dst_img.exists() and dst_img.samefile(src_img):
        pass
    elif not dst_img.exists():
        if use_symlinks:
            os.symlink(src_img.resolve(), dst_img)
        else:
            shutil.copy2(src_img, dst_img)
    if dst_lbl.exists() and dst_lbl.samefile(src_lbl):
        pass
    elif not dst_lbl.exists():
        if use_symlinks:
            os.symlink(src_lbl.resolve(), dst_lbl)
        else:
            shutil.copy2(src_lbl, dst_lbl)


def link_real_split(
    real_root: Path,
    split: str,
    out_root: Path,
    use_symlinks: bool,
) -> None:
    img_dir = real_root / "images" / split
    lbl_dir = real_root / "labels" / split
    out_img = out_root / "images" / split
    out_lbl = out_root / "labels" / split
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    for p in img_dir.glob("*"):
        if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
            continue
        stem = p.stem
        lbl = lbl_dir / f"{stem}.txt"
        if not lbl.is_file():
            continue
        link_pair(p, lbl, out_img, out_lbl, use_symlinks)


def write_output_yaml(
    real_yaml: Path, out_root: Path, train_split: str, yaml_basename: str
) -> None:
    with open(real_yaml, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    cfg["path"] = str(out_root.resolve())
    cfg["train"] = [f"images/{train_split}", "images/synth"]
    out_y = out_root / yaml_basename
    with open(out_y, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

In [5]:
from IPython.display import display


def run_balance(
    ruta_reales: str,
    ruta_sinteticas: str,
    dataset_final: str,
    real_split: str = "train",
    augmentation_ratio: float = 0.0,
    random_state: int = 42,
    use_symlinks: bool = True,
) -> None:
    rng = random.Random(random_state)
    real_root = Path(ruta_reales).expanduser().resolve()
    synth_root = Path(ruta_sinteticas).expanduser().resolve()
    out_root = Path(dataset_final).expanduser().resolve()

    yaml_path = real_root / data_yaml_name
    if not yaml_path.is_file():
        raise FileNotFoundError(f"No se encuentra {yaml_path}")

    id_to_name = load_id_to_name(yaml_path)
    class_ids = sorted(id_to_name.keys())

    labels_dir = real_root / "labels" / real_split
    if not labels_dir.is_dir():
        raise FileNotFoundError(f"No existe {labels_dir}")

    real_per_class, n_real_files = count_real_dataset(labels_dir)
    unknown_real = set(real_per_class.keys()) - set(class_ids)
    if unknown_real:
        raise ValueError(
            f"Hay class ids en etiquetas reales que no están en data.yaml 'names': {sorted(unknown_real)}"
        )
    for c in class_ids:
        real_per_class.setdefault(c, 0)

    M = max(real_per_class.values()) if real_per_class else 0
    needs_balance = {c: max(0, M - real_per_class[c]) for c in class_ids}

    pool = build_synth_pool(synth_root, prompts_csv_name)
    pool_by_idx = {s.idx: s for s in pool}

    forbidden: Set[int] = set()
    chosen_balance = apply_selection(pool, class_ids, needs_balance, forbidden, rng)

    synth_after_balance = summarize_from_images(pool_by_idx, chosen_balance, class_ids)
    total_after_balance = {c: real_per_class[c] + synth_after_balance.get(c, 0) for c in class_ids}

    chosen_extra: List[int] = []
    if augmentation_ratio > 0:
        extra_per_class = int(round(augmentation_ratio * M))
        if extra_per_class > 0:
            needs_extra = {c: extra_per_class for c in class_ids}
            chosen_extra = apply_selection(pool, class_ids, needs_extra, forbidden, rng)

    synth_extra = summarize_from_images(pool_by_idx, chosen_extra, class_ids)
    total_final = {
        c: real_per_class[c] + synth_after_balance.get(c, 0) + synth_extra.get(c, 0) for c in class_ids
    }

    # Informe
    print(f"Clases: {len(class_ids)}, M (max instancias reales/clase) = {M}")
    print(f"Imágenes reales ({real_split}): {n_real_files}")
    print(f"Sintéticas elegidas (balance): {len(chosen_balance)}")
    if augmentation_ratio > 0:
        print(
            f"Augmentation ratio={augmentation_ratio} -> +{int(round(augmentation_ratio * M))} inst/clase; "
            f"imágenes sint. extra: {len(chosen_extra)}"
        )

    short = [c for c in class_ids if total_after_balance[c] < M]
    if short:
        print("Aviso: tras balance no se alcanzó M en clases (falta de sintéticas o multi-etiqueta):", short)

    # Escribir salida
    out_root.mkdir(parents=True, exist_ok=True)
    (out_root / "images" / "synth").mkdir(parents=True, exist_ok=True)
    (out_root / "labels" / "synth").mkdir(parents=True, exist_ok=True)

    link_real_split(real_root, real_split, out_root, use_symlinks)

    all_synth = chosen_balance + chosen_extra
    dst_img_dir = out_root / "images" / "synth"
    dst_lbl_dir = out_root / "labels" / "synth"
    for idx in tqdm(all_synth, desc="Sintéticas -> out"):
        s = pool_by_idx[idx]
        link_pair(s.img_path, s.lbl_path, dst_img_dir, dst_lbl_dir, use_symlinks)

    write_output_yaml(yaml_path, out_root, real_split, data_yaml_name)

    df_report = pd.DataFrame(
        {
            "class_id": class_ids,
            "name": [id_to_name[c] for c in class_ids],
            "real": [real_per_class[c] for c in class_ids],
            "synth_balance": [synth_after_balance.get(c, 0) for c in class_ids],
            "synth_aug": [synth_extra.get(c, 0) for c in class_ids],
            "total": [total_final[c] for c in class_ids],
            "target_M": [M] * len(class_ids),
        }
    )
    display(df_report)


run_balance(
    ruta_reales,
    ruta_sinteticas,
    dataset_final,
    real_split=real_split,
    augmentation_ratio=augmentation_ratio,
    random_state=random_state,
    use_symlinks=use_symlinks,
)

Aviso: sintéticas omitidas — sin imagen: 0, sin etiqueta: 2000
Clases: 20, M (max instancias reales/clase) = 4194
Imágenes reales (train): 5717
Sintéticas elegidas (balance): 8467
Aviso: tras balance no se alcanzó M en clases (falta de sintéticas o multi-etiqueta): [2, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19]


Sintéticas -> out: 100%|██████████| 8467/8467 [00:02<00:00, 3783.18it/s]


,class_id,name,real,synth_balance,synth_aug,total,target_M
0,0,aeroplane,432,3762,0,4194,4194
1,1,bicycle,353,3841,0,4194,4194
2,2,bird,560,3585,0,4145,4194
3,3,boat,426,3768,0,4194,4194
4,4,bottle,629,3565,0,4194,4194
5,5,bus,292,3902,0,4194,4194
6,6,car,1013,3181,0,4194,4194
7,7,cat,605,0,0,605,4194
8,8,chair,1178,0,0,1178,4194
9,9,cow,290,0,0,290,4194
